# Practice 54 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — Remote team standups

A team holds daily standups. Each member dials in from their local timezone at the same UTC time.

- Add `local_time`: convert `standup_utc` to each member's `tz`.
- Add `local_hour`: the hour of the local standup time.
- Who has the earliest local standup hour? Who has the latest?
- Add `next_standup_utc`: the standup time 5 business days later. Use `pd.offsets.BusinessDay(5)`.
- Use NumPy to find the mean and std of `local_hour` across the team.

In [18]:
team = pd.DataFrame({
    'name': ['Alice', 'Bruno', 'Celine', 'Daisuke', 'Eva'],
    'tz':   ['US/Eastern', 'Europe/Berlin', 'US/Pacific', 'Asia/Tokyo', 'Australia/Sydney'],
    'standup_utc': pd.to_datetime([
        '2024-05-08 15:00', '2024-05-08 15:00', '2024-05-08 15:00',
        '2024-05-08 15:00', '2024-05-08 15:00',
    ]).tz_localize('UTC'),
})

# Your code here

team['local_time'] =team.apply(lambda row: row['standup_utc'].tz_convert(row['tz']), axis = 1)
team['local_hour'] = team['local_time'].apply(lambda x: x.hour)
print('earliest: ',team['name'][team['local_hour'].idxmin()])
print('latest: ',team['name'][team['local_hour'].idxmax()])
team['next_standup_utc'] = team['standup_utc'] + pd.offsets.BusinessDay(5)
print('mean: ',np.mean(team['local_hour']))
print(f'std is {np.std(team["local_hour"])}')

earliest:  Daisuke
latest:  Bruno
mean:  7.4
std is 6.343500610861483


---
## Level 2 — Monthly subscription revenue

Monthly revenue across three subscription tiers, Jan 2023 – Jun 2024. `month` is the index.

- For each tier, join last year's revenue using a freq-aware shift and compute `yoy_pct` (rounded to 1 decimal).
- Which tier had the highest mean YoY growth? Use NumPy, ignoring NaN.
- Stack the result into a long-format Series indexed by `(month, tier)`. Filter to months where every tier grew (all `yoy_pct > 0`).
- How many such months are there?

In [ ]:
np.random.seed(7)
idx = pd.date_range('2023-01-01', periods=18, freq='MS')
subs = pd.DataFrame({
    'basic':    np.random.randint(40000, 60000, 18),
    'pro':      np.random.randint(25000, 40000, 18),
    'enterprise': np.random.randint(10000, 20000, 18),
}, index=idx)
subs.index.name = 'month'

# Your code here


subs_shift = subs.shift(12, freq='MS')
subs_join = subs.join(subs_shift, lsuffix='_last_year')
for i in subs.columns:
    lastn = i + '_last_year'
    ncn = i+'_yoy_pct'
    subs_join[ncn] = round((subs_join[i]-subs_join[lastn])/subs_join[lastn]*100,1)


meang = {}
for t in subs.columns:
    meang[t] =np.nanmean(subs_join[t + '_yoy_pct']) 
list(meang.keys())[np.argmax(list(meang.values()))]

sel_cols = [c for c in subs_join.columns if 'yoy' in c]
subs_s = subs_join[sel_cols]

ss = subs_s.stack().rename_axis(['month','tier'])
ss_df = ss.to_frame('yoy')
month_res = ss_df.groupby(level = 'month')['yoy'].min()
[m for m in month_res.index if month_res[m]>0]

len([m for m in month_res.index if month_res[m]>0])

ValueError: Columns must be same length as key

In [95]:
yoy_cols + '_yoy_pct'

Index(['basic_yoy_pct', 'pro_yoy_pct', 'enterprise_yoy_pct'], dtype='object')

---
## Level 3 — Conference schedule

A conference runs sessions across three tracks. Session times are stored as UTC; attendees are in different timezones.

No prescribed steps.

1. Add `duration_hr`: session length in hours.
2. Add `end_utc`: `start_utc` + duration.
3. For each session, add `start_sf` (US/Pacific) and `start_berlin` (Europe/Berlin).
4. Which track has the highest mean duration? Use NumPy.
5. Use a list comprehension to collect session titles that start before 9am Pacific time.
6. Build a pivot table: rows = `track`, columns = `day` (the date of `start_utc`), values = total duration. Which track–day combination has the most content?

In [87]:
sessions = pd.DataFrame({
    'title': [
        'Keynote', 'ML Workshop', 'Data Pipelines', 'LLM Trends',
        'MLOps Deep Dive', 'Panel: AI Ethics', 'Vector DBs', 'Closing Talk',
    ],
    'track': ['Main', 'ML', 'Data', 'ML', 'MLOps', 'Main', 'Data', 'Main'],
    'start_utc': pd.to_datetime([
        '2024-09-10 16:00', '2024-09-10 17:30', '2024-09-10 19:00', '2024-09-10 21:00',
        '2024-09-11 15:00', '2024-09-11 17:00', '2024-09-11 19:30', '2024-09-11 21:00',
    ]).tz_localize('UTC'),
    'duration_min': [60, 90, 60, 45, 120, 60, 75, 30],
})

# Your code here

sessions['duration_hr'] = sessions['duration_min']/60
sessions['end_utc'] = sessions.apply(lambda row: row['start_utc'] + pd.DateOffset(hours = row['duration_hr']), axis = 1)

sessions['end_utc1'] = sessions['start_utc'] + pd.to_timedelta(sessions['duration_min'], unit='min')

sessions['start_sf'] = sessions['start_utc'].dt.tz_convert('US/Pacific')
sessions['start_berlin'] = sessions['start_utc'].dt.tz_convert('Europe/Berlin')
sessions.groupby('track')['duration_min'].mean().idxmax()
[t for t, s in zip(sessions['title'], sessions['start_sf']) if s.hour <9]
sessions['day'] = sessions['start_utc'].dt.day
pt = sessions.pivot_table(
    index = 'track',
    columns='day',
    values='duration_hr',
    aggfunc='sum'
)


pt.stack().idxmax()


('ML', np.int32(10))

In [88]:
sessions

,title,track,start_utc,duration_min,duration_hr,end_utc,end_utc1,start_sf,start_berlin,day
0,Keynote,Main,2024-09-10 16:00:00+00:00,60,1.00,2024-09-10 17:00:00+00:00,2024-09-10 17:00:00+00:00,2024-09-10 09:00:00-07:00,2024-09-10 18:00:00+02:00,10
1,ML Workshop,ML,2024-09-10 17:30:00+00:00,90,1.50,2024-09-10 19:00:00+00:00,2024-09-10 19:00:00+00:00,2024-09-10 10:30:00-07:00,2024-09-10 19:30:00+02:00,10
2,Data Pipelines,Data,2024-09-10 19:00:00+00:00,60,1.00,2024-09-10 20:00:00+00:00,2024-09-10 20:00:00+00:00,2024-09-10 12:00:00-07:00,2024-09-10 21:00:00+02:00,10
3,LLM Trends,ML,2024-09-10 21:00:00+00:00,45,0.75,2024-09-10 21:45:00+00:00,2024-09-10 21:45:00+00:00,2024-09-10 14:00:00-07:00,2024-09-10 23:00:00+02:00,10
4,MLOps Deep Dive,MLOps,2024-09-11 15:00:00+00:00,120,2.00,2024-09-11 17:00:00+00:00,2024-09-11 17:00:00+00:00,2024-09-11 08:00:00-07:00,2024-09-11 17:00:00+02:00,11
5,Panel: AI Ethics,Main,2024-09-11 17:00:00+00:00,60,1.00,2024-09-11 18:00:00+00:00,2024-09-11 18:00:00+00:00,2024-09-11 10:00:00-07:00,2024-09-11 19:00:00+02:00,11
6,Vector DBs,Data,2024-09-11 19:30:00+00:00,75,1.25,2024-09-11 20:45:00+00:00,2024-09-11 20:45:00+00:00,2024-09-11 12:30:00-07:00,2024-09-11 21:30:00+02:00,11
7,Closing Talk,Main,2024-09-11 21:00:00+00:00,30,0.50,2024-09-11 21:30:00+00:00,2024-09-11 21:30:00+00:00,2024-09-11 14:00:00-07:00,2024-09-11 23:00:00+02:00,11
